In [1]:
%pip install opencv-python numpy shapely matplotlib scikit-image tdmclient

Note: you may need to restart the kernel to use updated packages.


In [1]:
import cv2
import numpy as np
from shapely.geometry import Polygon
from tdmclient import ClientAsync

from vision import (initialize_camera, initialize_vision, detect_thymio, draw_thymio,
                    GridNotDetectedError, camera_id, camera_width, fps,
                    warmup_frames, grid_side_mm, project_to_camera)
from global_navigation import (build_free_space, compute_path,
                               draw_obstacles, draw_path, draw_legend)

from motion_control_test import follow_trajectory

ARENA_W = ARENA_H = grid_side_mm   # 700 mm, from vision (origin top left corner)


def init_obstacles(frame):
    """Run the vision, inflate the obstacles and show them on the frame.

    Returns H (pixels -> mm), to_camera (mm -> pixels), obstacles, inflated obstacles, goal.
    """
    H, obstacles_mm, goal_mm = initialize_vision(frame)
    to_camera = np.linalg.inv(H)
    obstacles = [Polygon(v) for v in obstacles_mm]
    _, inflated_obstacles, _ = build_free_space(obstacles, ARENA_W, ARENA_H)
    cv2.imshow("Vision: inflated obstacles",
               draw_legend(draw_obstacles(frame.copy(), obstacles, inflated_obstacles, to_camera)))
    return H, to_camera, obstacles, inflated_obstacles, goal_mm


async def set_speed(node, left, right):
    error = await node.set_variables({
        "motor.left.target": [left],
        "motor.right.target": [right],
    })
    if error is not None:
        raise RuntimeError(f"Thymio motor command failed: {error}")


async def main(node):
    cap = initialize_camera(camera_id, camera_width, fps, warmup_frames)
    frame = None

    try:
        await set_speed(node, 0, 0)
        dictionary = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
        detector = cv2.aruco.ArucoDetector(dictionary)
        # 1. Vision: grid, pieces, best move (once)
        ret, frame = cap.read()
        if not ret:
            raise RuntimeError("Cannot read the camera image")
        H, to_camera, obstacles, inflated_obstacles, goal_mm = init_obstacles(frame)

        waypoints = samples = None
        progress_index = 0
        arrived = False
        plan_pending = True
        while True:
            ret, frame = cap.read()
            if not ret:
                raise RuntimeError("Cannot read the camera image")

            # 2. Thymio pose (every frame)
            thymio = detect_thymio(frame, H, detector)
            print(thymio)

            # 3. Global navigation: once, as soon as the Thymio is seen
            if plan_pending and thymio is not None:
                waypoints, samples = compute_path(obstacles, thymio, goal_mm, ARENA_W, ARENA_H)
                plan_pending = False

            # 4. Motion control
            left, right, progress_index, target, reached = follow_trajectory(
                samples, thymio, progress_index
            )
            arrived = arrived or reached
            if arrived:
                left = right = 0
            await set_speed(node, left, right)

            # 5. Display the path, Thymio pose and active target
            view = draw_obstacles(frame.copy(), obstacles, inflated_obstacles, to_camera)
            if samples:
                view = draw_path(view, waypoints, samples, to_camera)
            draw_thymio(view, thymio, to_camera)
            if target is not None:
                target_px, = project_to_camera([target], to_camera)
                cv2.circle(view, tuple(map(int, target_px)), 7, (255, 0, 255), -1)
            cv2.imshow("Global navigation", draw_legend(view))

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):                 # quit
                break
            if key == ord("p") and thymio is not None:   # replan (e.g. after a kidnapping)
                await set_speed(node, 0, 0)
                waypoints, samples = compute_path(obstacles, thymio, goal_mm, ARENA_W, ARENA_H)
                progress_index = 0
                arrived = False
                plan_pending = False
            if key == ord("r"):                 # rerun the vision (pieces moved)
                await set_speed(node, 0, 0)
                try:
                    H, to_camera, obstacles, inflated_obstacles, goal_mm = init_obstacles(frame)
                    waypoints = samples = None
                    progress_index = 0
                    arrived = False
                    plan_pending = True
                except GridNotDetectedError as error:
                    print(error)

    except GridNotDetectedError as error:
        print(error)
        cv2.imshow("Grid not detected (press a key to quit)", frame)
        cv2.waitKey(0)
    finally:
        try:
            await set_speed(node, 0, 0)
        finally:
            cap.release()
            cv2.destroyAllWindows()


if __name__ == "__main__":
    with ClientAsync() as client:
        with await client.lock() as node:
            await main(node)

CancelledError: 